# 04 · 模組的第三種來源，跟換掉整個後端模型

> **目標**：30 分鐘。不開網頁，全部用程式碼跑一次 —— 接一個外部 MCP server，
> 讓它變成架構裡的一個模組；然後把後端模型換成別家的，證明**模組和 policy 一個字都不用改**。
>
> 前置：跑過 `02_compose_architectures.ipynb`。這本用同一份 `modules.py`。

## 這本要打掉的誤會

學生學完 02 之後通常會有兩個問題：

1. 「模組是不是都要我自己寫？」
2. 「我公司不能用 Anthropic，這套是不是就沒用了？」

兩題的答案都是**不用改架構**。這本就是把這兩件事跑給你看。

## Step 0 · 先看清楚：我們自己就是一個 MCP server

這件事比它聽起來重要。打開 `modules.py` 的 `build_mcp_server()`：

In [1]:
import os, sys, json, inspect, asyncio
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))
from dotenv import load_dotenv
load_dotenv()

import modules as M

print(inspect.getsource(M.build_mcp_server))

def build_mcp_server(arch: Architecture, ix: Index):
    """把架構用到的模組包成 in-process MCP server 交給 SDK。

    關鍵：不同架構拿到的工具清單不同 —— Naive RAG 只有 search，
    連「評估檢索品質」的能力都沒有，所以它結構上不可能自我修正。
    """
    from claude_agent_sdk import create_sdk_mcp_server, tool

    def make_handler(module_name: str):
        async def handler(args: dict) -> dict:
            result = run_module(module_name, args, ix)
            return {"content": [{"type": "text", "text": json.dumps(result, ensure_ascii=False)}]}

        return handler

    sdk_tools = [
        tool(name, MODULES[name]["description"], MODULES[name]["schema"])(make_handler(name))
        for name in arch.modules
        if name in MODULES
    ]
    return create_sdk_mcp_server("ragmod", "1.0.0", sdk_tools)



最後一行就是重點：

```python
return create_sdk_mcp_server("ragmod", "1.0.0", sdk_tools)
```

**我們那七個模組，從第一天就是包成一個叫 `ragmod` 的 MCP server 餵給 SDK 的。**
所以你在 02 學的「寫一個 `@tool`」，做出來的東西跟別人在 GitHub 上發布的 MCP server
是同一種東西 —— 只是住在同一個行程裡。

看一下 SDK 實際收到什麼：

In [2]:
from retrieval import load_index
# 跟 02 一樣：索引建在 data/，embedding 掛掉會自動降級成純 BM25
ix = load_index(Path("data"), embedding=os.getenv("EMBEDDING", "local"))

options, _ = M.build_options(M.ARCHITECTURES["crag"], ix)
print("mcp_servers 的 key ：", list(options.mcp_servers))
print("allowed_tools      ：")
for t in options.allowed_tools:
    print("   ", t)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Building prefix dict from the default dictionary ...


Loading model from cache /var/folders/d3/00wx88zj1d17mtmk5gj09vpm0000gn/T/jieba.cache


Loading model cost 0.193 seconds.


Prefix dict has been built successfully.


mcp_servers 的 key ： ['ragmod']
allowed_tools      ：
    mcp__ragmod__search
    mcp__ragmod__grade_documents
    mcp__ragmod__expand
    WebSearch


`mcp__ragmod__search` 前面那兩段就是「哪一個 server」「哪一個工具」。
**外部 MCP server 接進來，長得一模一樣，只是中間那段換成別人的名字。**

## Step 1 · 接一個外部 MCP server

`mcp_registry.probe()` 會連上去、等它從 `pending` 變 `connected`、
把工具清單抓回來。這裡用 Context7（查套件官方文件，不需要 API key）。

> 教室沒網路的話這格會失敗 —— 那正好，下一格就是示範失敗長什麼樣。

In [3]:
import mcp_registry

CONFIG = {"type": "http", "url": "https://mcp.context7.com/mcp"}
got = await mcp_registry.probe(CONFIG)

print("狀態：", got["status"])
print("是誰：", got["info"].get("name"), got["info"].get("version"))
print("它說自己做什麼：", got["info"].get("description", "")[:80])
print("它提供的工具：")
for t in got["tools"]:
    print("   mcp__context7__" + t)

狀態： connected
是誰： Context7 4.1.1
它說自己做什麼： Context7 provides up-to-date documentation and code examples for libraries and f
它提供的工具：
   mcp__context7__resolve-library-id
   mcp__context7__query-docs


### 連不上長什麼樣

這格是刻意要失敗的。**連不上就不給存**是這一層的設計 ——
存一個死的 server，下次跑不動時你只會以為是自己的 policy 寫壞了。

In [4]:
bad = await mcp_registry.probe({"command": "this-command-does-not-exist-xyz"})
print("狀態：", bad["status"])
print("錯誤：", bad["error"])

狀態： failed
錯誤： Executable not found in $PATH: "this-command-does-not-exist-xyz"


## Step 2 · 讓它變成架構裡的一個模組

`Architecture` 多一個 `mcp_tools` 欄位，裡面放**完整名稱**。
就這樣 —— 沒有第二套機制，它跟 `modules` 走同一條路。

In [5]:
# 登記起來，build_options 才掛得上這個 server
await mcp_registry.add("context7", CONFIG, stage="Retrieval")

外援 = M.Architecture(
    name="本地查不到才問官方文件",
    paper="示範：模組可以來自別人的 MCP server",
    orchestration="Conditional（本地優先，缺了才走外部）",
    modules=["search", "grade_documents"],
    mcp_tools=["mcp__context7__resolve-library-id", "mcp__context7__query-docs"],
    max_turns=10,
    policy="""第 1 步：一律先用 search 查本地知識庫。

第 2 步：用 grade_documents 讀完整內文逐塊評分（2=直接答得出來、1=沾邊、0=無關）。

分流：
- 有 2 分片段 → 直接作答，出處用 [檔名#編號]。
- 全部低於 1 分 → 判定這題不在知識庫裡，改走外部：先用
  mcp__context7__resolve-library-id 把題目裡的套件名換成它的 library id，
  再用 mcp__context7__query-docs 查文件。

作答規則：本地片段的出處寫 [檔名#編號]，外部 MCP 拿回來的寫 [mcp: 工具名]，
兩種**分開標**，不可以混。開頭先用一句話講清楚這題是本地答的還是外部答的。""",
)

opts, _ = M.build_options(外援, ix)
print("這次會連上的 server：", list(opts.mcp_servers))
print("白名單：")
for t in opts.allowed_tools:
    print("   ", t)

這次會連上的 server： ['ragmod', 'context7']
白名單：
    mcp__ragmod__search
    mcp__ragmod__grade_documents
    mcp__context7__resolve-library-id
    mcp__context7__query-docs


注意 `mcp_servers` 只有 `ragmod` 和 `context7` —— **只掛這個架構真的用到的**。
掛了沒用到的有兩個代價：連線成本（stdio 要開行程），以及偷偷擴大授權。

再看 system prompt 尾巴自動長出來的那段：

In [6]:
print(外援.system_prompt()[-330:])

__resolve-library-id 把題目裡的套件名換成它的 library id，
  再用 mcp__context7__query-docs 查文件。

作答規則：本地片段的出處寫 [檔名#編號]，外部 MCP 拿回來的寫 [mcp: 工具名]，
兩種**分開標**，不可以混。開頭先用一句話講清楚這題是本地答的還是外部答的。

## 這次還接了外部 MCP 工具

- `mcp__context7__resolve-library-id`
- `mcp__context7__query-docs`

它們**不是知識庫的一部分**。引用它們回來的內容時，出處要標成 `[mcp: <工具名>]`，不要混進 `[檔名#編號]` 那種知識庫出處。


出處分開標是硬規則。RAG 的價值是「每一句話都查得到出處」——
外部內容混進來又標成知識庫格式，**使用者就分不出哪一句有語料撐腰**。
那比沒有出處更危險，因為它看起來很可信。

## Step 3 · 真的跑一題

故意問一個知識庫（Claude Code 教材）絕對沒有的東西。

In [7]:
run = await M.arun(外援, "FastAPI 的 BackgroundTasks 要怎麼用？", ix)
print()
print("=" * 60)
print(run.answer[:700])

  [1] search({"query": "FastAPI BackgroundTasks", "method": "bm25", "k": 5})
       → 0 筆


  [2] search({"query": "如何在 API 中執行背景任務、非同步處理回應後的工作", "method": "vector", "k": 5})


       → 5 筆


  · 先照流程走：這題問的是 FastAPI 的具體 API 名稱（BackgroundTasks），關鍵字精確匹配會最準，但也怕知識庫是用「背景任務」之類的說法描述，所以我同時跑 bm25 和 vector 兩路。


  [3] grade_documents({"question": "FastAPI 的 BackgroundTasks 要怎麼用？", "chunk_ids": ["01-hooks.md#1", "04-skills.md#2", "09-rag-patte)
       → 取回 4 塊完整內文


...

  · 評分結果：`01-hooks.md#1` = 0、`04-skills.md#2` = 0、`09-rag-patterns.md#2` = 0、`07-agent-sdk.md#3` = 0。這份知識庫是 Claude Code / Agentic RAG 的工作坊教材，完全不含 FastAPI 


  [5] mcp__context7__query-docs({"libraryId": "/websites/fastapi_tiangolo", "query": "How to use BackgroundTasks: add_task, using it with depe)


       → ### Use BackgroundTasks with Dependency Injection in FastAPI

Source: https://fastapi.tiangolo.com/tutorial/background-tasks

Declare BackgroundTasks in both pa


  · 官方文件站點分數與覆蓋率最高，選 `/websites/fastapi_tiangolo` 查 BackgroundTasks 的用法。



**這題是外部答的**——本地知識庫（Claude Code / Agentic RAG 教材）完全沒有 FastAPI 內容，bm25 零命中、vector 撈回的四塊評分全是 0 分，所以轉去查 Context7 官方文件。

## 基本用法

在路徑操作函式裡宣告一個 `BackgroundTasks` 型別的參數，再用 `.add_task()` 註冊，任務會在**回應送出給客戶端之後**才執行 [mcp: mcp__context7__query-docs]：

```python
from fastapi import BackgroundTasks, FastAPI

app = FastAPI()


def write_notification(email: str, message=""):
    with open("log.txt", mode="w") as email_file:
        content = f"notification for {email}: {message}"
        email_file.write(content)


@app.post("/send-notification/{email}")
async def send_notification(email: str, background_tasks: BackgroundTasks):
    background_tasks.add_task(write_notification, email, message="some notificat


軌跡會長這樣：本地 `search` 撈不到 → `grade_documents` 全判 0 分 →
才轉去 `mcp__context7__*`。**評分模組是那個把關的人** ——
沒有它，第一輪撈回來的不相關片段就會被拿去作答。

答案開頭它自己會交代這題是外部答的，出處也分開標了。

## Step 4 · 換掉整個後端模型

SDK 是去 spawn `claude` 這支 CLI，所以「用哪個模型」不是程式碼的事，
是**環境變數**的事。而 `ClaudeAgentOptions.env` 是**每次 spawn 才疊上去**的 ——
所以可以每一次呼叫都換一個供應商，不用重開任何東西。

In [8]:
import providers

print("現在用的是：", providers.status()["name"])
print("哪裡來的　：", providers.status()["source"])
print()
print("可以切的：")
for p in providers.PRESETS:
    need = "、".join(f["label"] for f in p["fields"]) or "什麼都不用填"
    print(f"  {p['key']:<11} {p['name']:<28} 要填：{need}")

現在用的是： OAuth 訂閱（本機 claude 已登入）
哪裡來的　： 本機 claude 登入狀態

可以切的：
  oauth       OAuth 訂閱（本機 claude 已登入）      要填：什麼都不用填
  anthropic   Anthropic API Key            要填：API key、模型（選填）
  compatible  Anthropic 相容端點               要填：Base URL、Token、模型
  bedrock     AWS Bedrock                  要填：Region、模型 ID（選填）
  vertex      Google Vertex AI             要填：Region、GCP 專案 ID、模型 ID（選填）


### 切過去會發生什麼事

`env_overlay()` 就是會疊在子行程環境上的那包東西。
注意它會把**沒用到的變數清成空字串** —— env 是只加不減的，
不清就會被上一個供應商的殘留汙染。

In [9]:
providers.use("compatible", {
    "ANTHROPIC_BASE_URL": "http://127.0.0.1:9",   # 故意指向一個不會有人聽的埠
    "ANTHROPIC_AUTH_TOKEN": "not-a-real-token",
    "ANTHROPIC_MODEL": "fake-model-x",
})
for k, v in providers.env_overlay().items():
    print(f"  {k:<28} = {v!r}")
print()
print("model 覆寫：", providers.model_override())

  ANTHROPIC_API_KEY            = ''
  ANTHROPIC_AUTH_TOKEN         = 'not-a-real-token'
  ANTHROPIC_BASE_URL           = 'http://127.0.0.1:9'
  ANTHROPIC_MODEL              = 'fake-model-x'
  CLAUDE_CODE_USE_BEDROCK      = ''
  CLAUDE_CODE_USE_VERTEX       = ''
  AWS_REGION                   = ''
  CLOUD_ML_REGION              = ''
  ANTHROPIC_VERTEX_PROJECT_ID  = ''

model 覆寫： fake-model-x


### 為什麼一定要有「試連看看」

打錯 base URL 的話，**CLI 不會報錯，它會安靜地一直重試**。
問答那邊就只是轉圈圈 —— 學生第一反應一定是「我 policy 是不是寫壞了」，
然後花二十分鐘找錯地方。

`providers.test()` 發一句「回答 OK」，60 秒沒回應就明講問題在哪。

In [10]:
print(await providers.test())

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors
[claude-code:unrecognized_model] {"model":"fake-model-x","query_source":"sdk"}


{'ok': False, 'error': '60 秒內沒有回應。base URL 或 token 不對的話 CLI 會安靜地一直重試 —— 先檢查這兩個。'}


In [11]:
providers.reset()
print("還原：", providers.status()["name"])
print(await providers.test())

還原： OAuth 訂閱（本機 claude 已登入）


{'ok': True, 'text': 'OK', 'active': {'key': 'oauth', 'name': 'OAuth 訂閱（本機 claude 已登入）', 'source': '本機 claude 登入狀態', 'values': {}}}


**整本 notebook 到這裡，`modules.py` 一個字都沒改過。**
換供應商改的是環境變數，不是你的架構 —— 你的架構本來就不知道它背後是誰。

## Step 5 · policy 跟圖是同一件事的兩種樣子

`/studio` 那頁可以把畫布編譯成 policy，也可以反過來把 policy 讀成圖。
反推那一支就是 `architect.graph_from_policy()`，這裡不開網頁直接叫它。

拿 CRAG 來試 —— 看它能不能還原出論文的三個分支。

In [12]:
import architect

crag = M.ARCHITECTURES["crag"]
g = await architect.graph_from_policy(crag.policy, crag.modules, crag.builtin_tools, [])

name = lambda n: {"__start__": "問題進來", "__answer__": "產生答案"}.get(n, n)
for e in g["edges"]:
    cond = f'  「{e["label"]}」' if e["label"] else ""
    print(f'{name(e["from"]):>16} → {name(e["to"]):<16}{cond}')

            問題進來 → search          
          search → grade_documents 
 grade_documents → 產生答案              「CORRECT（多數 2 分）→ decompose-then-recompose 後作答」
 grade_documents → expand            「CORRECT（多數 2 分）且片段被切斷」
          expand → 產生答案            
 grade_documents → WebSearch         「INCORRECT（多數 0 分）→ 丟掉這批檢索結果，改用 WebSearch 上網找」
 grade_documents → WebSearch         「AMBIGUOUS（介於中間、判斷不了）→ 兩邊都用：精煉後的內部知識＋WebSearch 的外部結果」
       WebSearch → 產生答案            


三個分支（Correct / Incorrect / Ambiguous）應該都在，而且條件字是從 policy 原文抓的。

> **這就是這一系列的主張收尾**：圖可以編譯成 policy，policy 也可以反推回圖，
> 因為它們講的是同一件事。**但被執行的永遠是文字那一份。**

## 收尾

| 這本證明了什麼 | 怎麼證的 |
|---|---|
| 模組不一定要你自己寫 | 接了 Context7，它的工具跟 `search` 一樣只是 `allowed_tools` 裡的一個名字 |
| 外部來源要看得出來 | `mcp_tools` 一填，system prompt 自動要求 `[mcp: 工具名]` 跟 `[檔名#編號]` 分開標 |
| 後端模型可以換 | `env_overlay()` 疊在子行程環境上，`modules.py` 一個字沒改 |
| 圖和 policy 等價 | `graph_from_policy()` 把 CRAG 的 policy 還原成論文的三個分支 |

**下一步**：把你自己已經在用的 MCP server 接進來（`mcp_registry.importable()`
會讀 `~/.claude.json` 列出來），組一個架構，跟 Naive RAG 並排跑一題。